<a href="https://colab.research.google.com/github/vadluri-dineshwar/generative-ai-lstm-text-generation/blob/main/JenyaInterviewTask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import urllib.request

1. DATASET LOADING AND PREPROCESSING

In [8]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "shakespeare.txt"

def download_dataset():
    """Download the Tiny Shakespeare dataset if not already present locally."""
    if not os.path.exists(DATA_PATH):
        print("Downloading dataset...")
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        text = f.read()
    return text

text = download_dataset()
text = text.lower()
print(f"Corpus length: {len(text)} characters")

vocab = sorted(set(text))
vocab_size = len(vocab)
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for i, ch in enumerate(vocab)}
print(f"Vocabulary size: {vocab_size} unique characters")

encoded = np.array([char_to_idx[c] for c in text])

SEQ_LENGTH = 100
STEP = 3

sequences = []
next_chars = []
for i in range(0, len(encoded) - SEQ_LENGTH, STEP):
    sequences.append(encoded[i:i + SEQ_LENGTH])
    next_chars.append(encoded[i + SEQ_LENGTH])

X = np.array(sequences)
y = np.array(next_chars)
print(f"Number of training sequences: {len(X)}")

split_idx = int(len(X) * 0.9)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

Corpus length: 1115394 characters
Vocabulary size: 39 unique characters
Number of training sequences: 371765


2. MODEL DESIGN

In [9]:
EMBED_DIM = 64
LSTM_UNITS = 256

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, input_length=SEQ_LENGTH),
    LSTM(LSTM_UNITS, return_sequences=True),
    LSTM(LSTM_UNITS),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

 3. MODEL TRAINING

In [10]:
EMBED_DIM = 64
LSTM_UNITS = 256

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, input_length=SEQ_LENGTH),
    LSTM(LSTM_UNITS, return_sequences=True),
    LSTM(LSTM_UNITS),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

4. TEXT GENERATION

In [11]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    ModelCheckpoint("best_model.keras", monitor="val_loss", save_best_only=True),
]

EPOCHS = 10
BATCH_SIZE = 256

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
)

Epoch 1/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 79s 55ms/step - accuracy: 0.3569 - loss: 2.2114 - val_accuracy: 0.4295 - val_loss: 1.8925
Epoch 2/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 75s 58ms/step - accuracy: 0.4957 - loss: 1.6765 - val_accuracy: 0.4826 - val_loss: 1.7131
Epoch 3/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 77s 59ms/step - accuracy: 0.5382 - loss: 1.5114 - val_accuracy: 0.5058 - val_loss: 1.6366
Epoch 4/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 79s 61ms/step - accuracy: 0.5613 - loss: 1.4221 - val_accuracy: 0.5158 - val_loss: 1.5926
Epoch 5/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 80s 61ms/step - accuracy: 0.5773 - loss: 1.3589 - val_accuracy: 0.5225 - val_loss: 1.5748
Epoch 6/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 82s 62ms/step - accuracy: 0.5903 - loss: 1.3097 - val_accuracy: 0.5320 - val_loss: 1.5606
Epoch 7/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 80s 61ms/step - accuracy: 0.6020 - loss: 1.2658 - val_accuracy: 0.5339 - val_loss: 1.5538
Epoch 8/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 79s 60ms/step - accuracy: 0.6125 -

Generation function + sample outputs

In [12]:
def sample_with_temperature(preds, temperature=1.0):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)


def generate_text(seed, length=400, temperature=0.8):
    seed = seed.lower()
    if len(seed) < SEQ_LENGTH:
        seed = " " * (SEQ_LENGTH - len(seed)) + seed
    else:
        seed = seed[-SEQ_LENGTH:]

    generated = seed
    current_seq = [char_to_idx.get(c, 0) for c in seed]

    for _ in range(length):
        x_pred = np.array(current_seq[-SEQ_LENGTH:]).reshape(1, SEQ_LENGTH)
        preds = model.predict(x_pred, verbose=0)[0]
        next_idx = sample_with_temperature(preds, temperature)
        next_char = idx_to_char[next_idx]
        generated += next_char
        current_seq.append(next_idx)

    return generated


seeds = [
    "to be, or not to be",
    "romeo, romeo, wherefore art thou",
    "friends, romans, countrymen",
]

for seed in seeds:
    print(f"\n--- Seed: '{seed}' ---")
    print(generate_text(seed, length=300, temperature=0.8))


--- Seed: 'to be, or not to be' ---
                                                                                 to be, or not to be draw'st.
and your hand to his great duke;
breath it, i shall broak hoith a couse you.

clarence:
nay, some barks abrace! turn thy stones,
toward the partics a regirent mantrous,
but, thou not in his soul, a spead of state;
but shall be care, so long and from him, here,
and shall both his ask a gurt

--- Seed: 'romeo, romeo, wherefore art thou' ---
                                                                    romeo, romeo, wherefore art thou
art the new.

first lord:
so, what is a poor lord, which you see the name,
must not have blose please him charge of our hand.

king richard ii:
sow i pracued me, mortality,
against a both and know of king edward's sull'd
both with these tituge of a heart.

buckingham:
o thank you, i can a very lord

--- Seed: 'friends, romans, countrymen' ---
                                                                  

Bonus temperature experiment

In [13]:
for temp in [0.5, 1.0, 1.5]:
    print(f"\n--- Temperature: {temp} ---")
    print(generate_text("to be, or not to be", length=200, temperature=temp))


--- Temperature: 0.5 ---
                                                                                 to be, or not to be
the like and dost thou that be deceived the city,
that he did shall be confering now.

paulina:
i had a volsces hold here.

coriolanus:
that i would not shall be the bloody looks.

king richard ii:
s

--- Temperature: 1.0 ---
                                                                                 to be, or not to be
thee, our drunkless day's wisdal hands:
could so shall shall just to see at:
and thou your his dear of this prince is prolible;
that i mis-dread o'er i contenment
here afterd'd my brother, 'tis the f

--- Temperature: 1.5 ---
                                                                                 to be, or not to be
anrocher by mnjiit now of cohit.
horse far bear him, isabelarland is deed godreit. flies,
dea-fell'n, peace?

northumberland
well,
no, you lot twois ablamour? ah, i'llly pod
aboll rake usjredance.

w
